#📌 Extracão

In [ ]:
import pandas as pd
import json

# Lendo o arquivo JSON
with open('TelecomX_Data.json', 'r') as f:
    dados_brutos = json.load(f)

# Transformando o JSON aninhado em um DataFrame plano (flat)
df = pd.json_normalize(dados_brutos)

# Visualizando as primeiras 5 linhas e as colunas geradas
print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.")
df.head()

#🔧 Transformação

In [ ]:
# 1. Verificando os tipos de dados iniciais
print(df.dtypes)

# 2. Convertendo account.Charges.Total para numérico
# O parâmetro errors='coerce' transforma espaços vazios em NaN (nulos)
df['account.Charges.Total'] = pd.to_numeric(df['account.Charges.Total'], errors='coerce')

# 3. Tratando valores nulos
# Vamos ver quantos nulos surgiram na conversão (geralmente clientes com tenure 0)
print(f"Valores nulos encontrados: {df.isnull().sum().sum()}")
df.dropna(inplace=True) # Removendo linhas com valores nulos para simplificar

# 4. Traduzindo/Renomeando colunas para facilitar a análise (opcional, mas recomendado)
df.columns = [col.replace('.', '_') for col in df.columns]

# 5. Verificando se há dados duplicados
df.drop_duplicates(inplace=True)

print("Transformação concluída!")

#📊 Carga e análise

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurando o estilo dos gráficos
sns.set_theme(style="whitegrid")

# Visualização 1: Distribuição da variável alvo (Churn)
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Churn', palette='viridis')
plt.title('Distribuição de Clientes: Retidos vs. Evasão (Churn)')
plt.show()

# Visualização 2: Churn por tipo de Contrato
# Este é um dos maiores causadores de evasão em Telecom
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='account_Contract', hue='Churn', palette='magma')
plt.title('Influência do Tipo de Contrato na Evasão')
plt.xlabel('Tipo de Contrato')
plt.ylabel('Quantidade de Clientes')
plt.show()

# Visualização 3: Gastos Mensais vs Churn
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Churn', y='account_Charges_Monthly', palette='Set2')
plt.title('Relação entre Gastos Mensais e Churn')
plt.show()

#📄Relatorio Final

**Relatório de Análise: Evasão de Clientes (Churn) - Telecom X**

**Analista:** Marcelo

**Projeto:** Programa Oracle Next Education (ONE) / Alura

**1. Introdução e Objetivos**
Este relatório apresenta o processo de ETL (Extração, Transformação e Carga) e a Análise Exploratória (EDA) dos dados da Telecom X. O objetivo principal é identificar os padrões que levam ao Churn (cancelamento de contrato por parte dos clientes), fornecendo uma base sólida para que a equipe de Data Science possa criar modelos preditivos de retenção.

**2. Processo de ETL (Extract, Transform, Load)**
**2.1 Extração**
Os dados foram extraídos de uma API disponibilizada via repositório no GitHub em formato JSON. Devido à estrutura aninhada dos dados (objetos de cliente, conta e internet dentro de cada registro), utilizei a biblioteca pandas com a função json_normalize.

Resultado: Transformação de um arquivo semi-estruturado em um DataFrame tabular de 7267 linhas, permitindo a análise estatística.

**2.2 Transformação (Limpeza e Tratamento)**
Para garantir a qualidade da análise, foram realizadas as seguintes etapas de limpeza:

Tipagem de Dados: A coluna Charges.Total (Gasto Total) foi convertida de texto para numérico. Identifiquei que valores vazios ocorriam em clientes novos (tempo de contrato zero), os quais foram tratados para não enviesar a média.

Padronização: Renomeação das colunas para remover pontos e facilitar a manipulação no Python.

Remoção de Duplicados e Nulos: Limpeza de registros inconsistentes para garantir a integridade dos insights.

**3. Análise Exploratória de Dados (EDA)**
A partir das visualizações geradas com Seaborn e Matplotlib, identificamos três padrões críticos:

Impacto do Contrato: Clientes com contrato mês a mês (Month-to-month) possuem uma taxa de evasão drasticamente superior aos clientes de contratos anuais.

Serviços Adicionais: A ausência de Suporte Técnico (Tech Support) e Segurança Online está fortemente correlacionada com a saída do cliente.

Gastos Mensais: Clientes com churn tendem a ter faturas mensais mais altas do que a média dos clientes fidelizados, sugerindo uma percepção de "baixo custo-benefício".

**4. Conclusão e Recomendações**
A análise demonstra que a evasão na Telecom X não é aleatória, mas sim concentrada em perfis de contrato de curto prazo e clientes sem suporte dedicado.

**Sugestões para Redução de Churn:**
**Incentivo à Fidelização:** Criar campanhas de marketing para migrar clientes "mês a mês" para planos anuais, oferecendo descontos progressivos.

**Pacotes de Suporte:** Oferecer o serviço de "Tech Support" como cortesia por 3 meses para clientes novos, visando aumentar o engajamento inicial.

**Monitoramento de Gastos:** Implementar um alerta para a equipe de sucesso do cliente quando um usuário atingir faturas 20% acima da média de seu perfil, permitindo uma renegociação proativa.